# Long-Run Risk Pricing via Ergodic BSDEs

The **long-run risk** model of Bansal & Yaron (2004) captures the premium
investors demand for exposure to *long-horizon* risks.

Under **Epstein-Zin** recursive preferences with risk aversion $\gamma$ and
elasticity of intertemporal substitution $\psi$, the continuation value
satisfies an ergodic BSDE whose **ergodic constant $\lambda$** equals the
*long-run risk-adjusted discount rate*.

The driver (log-linearised around the stationary mean of the OU proxy $X_t$):

$$
f(x, v, z) = \delta\,\theta_{\rm EZ}\,x + \frac{\gamma}{2}|z|^2,
\quad \theta_{\rm EZ} = \frac{1-\gamma}{1 - 1/\psi},
$$

where $\delta$ is the subjective discount rate.

**Pricing implication**: the price-dividend ratio satisfies

$$
P_t / D_t \longrightarrow e^{-(\lambda - g)T} \quad \text{as } T \to \infty,
$$

so a higher $\lambda$ implies a *lower* long-run price-dividend ratio.


In [ ]:
import numpy as np
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
%matplotlib inline

from ebsde.applications.long_run_risk import LongRunRiskPricer


## Epstein-Zin Preferences

The Epstein-Zin aggregator coefficient

$$
\theta_{\rm EZ} = \frac{1-\gamma}{1 - 1/\psi}
$$

captures how risk aversion and intertemporal substitution interact.

- $\gamma > 1/\psi$ (standard calibration): $\theta_{\rm EZ} < 0$,
  so agents prefer early resolution of uncertainty → long-run risk premium.
- $\gamma = 1/\psi$ (CRRA): $\theta_{\rm EZ}$ is degenerate.


In [ ]:
# --- Calibrate & solve ---------------------------------------------
pricer = LongRunRiskPricer(gamma=5.0, psi=1.5, delta=0.02)

print(f"theta_EZ = {pricer.theta_ez:.4f}  "
      f"(gamma={pricer.gamma}, psi={pricer.psi})")

results = pricer.compute_risk_adjusted_rate(method='pde')

print()
print(f"Long-run risk-adjusted rate  lambda  = {results['lambda']:.6f}")
print(f"Risk premium (lambda - E[f])          = {results['risk_premium']:.6f}")
print(f"Equity risk premium (approx annual %) = {100*results['equity_premium']:.2f}%")


## Sensitivity to Risk Aversion $\gamma$

Higher risk aversion $\gamma$ → steeper quadratic penalty on $z$ → larger
ergodic constant $\lambda$ (higher discount rate = lower equity valuations).


In [ ]:
import os
# --- Sensitivity analysis ------------------------------------------
gamma_values = [2.0, 5.0, 8.0]
df = pricer.sensitivity_analysis('gamma', gamma_values)

print(df.to_string(index=False))

# --- Plot ----------------------------------------------------------
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(df['gamma'], df['lambda'], 'o-', lw=2, ms=8, color='C3')
ax.set_xlabel('Risk aversion γ')
ax.set_ylabel('λ  (long-run discount rate)')
ax.set_title('Long-run risk premium vs risk aversion')
ax.grid(alpha=0.3)
fig.tight_layout()

os.makedirs('notebooks/figures', exist_ok=True)
fig.savefig('notebooks/figures/05_sensitivity.png', dpi=120)
plt.close(fig)
print("Figure saved → notebooks/figures/05_sensitivity.png")


## Term Structure of Risk

For finite maturity $T$ the yield $y(T) = -Y_0(T)/T$ traces the term structure.
As $T \to \infty$ the yield converges to the ergodic constant $\lambda$.

A *upward-sloping* term structure (yields rise with $T$) is consistent with a
positive long-run risk premium.


In [ ]:
# --- Term structure ------------------------------------------------
maturities = [0.5, 1.0, 2.0]
ts = pricer.term_structure_of_risk(maturities)

print(f"{'Maturity T':>12}  {'Yield y(T)':>12}")
print("-" * 27)
for T_val, y_val in zip(ts['maturities'], ts['yields']):
    print(f"{T_val:>12.2f}  {y_val:>12.6f}")
print(f"{'lambda (T->inf)':>12}  {ts['lambda_limit']:>12.6f}")
